In [1]:
 """
Bias-Variance Trade-off in Text Classification  (AG News, TF-IDF + linear models)
=================================================================================
ONE script that reproduces everything in the mini project report:
  * all experiments  (Tables 4.1 - 4.6)      -> out/results.json
  * all figures      (Figures 1.1 - 4.5)     -> out/fig_*.png
  * confusion matrix + per-class metrics     -> out/fig_cm.png, out/final.json
  * all result tables printed to the screen

Usage (Google Colab or terminal):
    python bias_variance_text_classification.py              # full run (~10-12 min on 1 CPU)
    python bias_variance_text_classification.py --use-saved  # skip experiments, reuse out/results.json

The AG News files (train.csv / test.csv) are downloaded automatically if missing.
Random seed is fixed (42) and the run order is fixed, so results are repeatable.
(Tiny differences are possible if your scikit-learn version differs from 1.8.)
"""
import os, sys, json, time, warnings, urllib.request
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import confusion_matrix, classification_report
warnings.filterwarnings("ignore")          # hides L-BFGS iteration-limit warnings
os.makedirs("out", exist_ok=True)

# ----------------------------------------------------------------- 1. DATA
URL = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/"
for f in ("train.csv", "test.csv"):
    if not os.path.exists(f):
        print("Downloading", f); urllib.request.urlretrieve(URL + f, f)
rd = lambda f: pd.read_csv(f, header=None, names=["y", "t", "d"])
tr, te = rd("train.csv"), rd("test.csv")
text = lambda d: (d.t + " . " + d.d).str.replace("\\", " ", regex=False).values
Xtr_text, Xte_text = text(tr), text(te)
ytr, yte = tr.y.values - 1, te.y.values - 1
print(f"Train: {len(tr)}  Test: {len(te)}")

# ----------------------------------------------------------------- 2. EXPERIMENTS
SIZES = [100, 200, 500, 1000, 2000, 5000, 10000, 20000, 50000, 120000]
REPS  = {100: 20, 200: 20, 500: 15, 1000: 10, 2000: 8, 5000: 5, 10000: 3, 20000: 3, 50000: 2, 120000: 1}

def make_model(name, C=1.0):
    return {"LogReg":     lambda: LogisticRegression(C=C, max_iter=100),
            "LinearSVM":  lambda: LinearSVC(C=C),
            "NaiveBayes": lambda: MultinomialNB(alpha=0.1)}[name]()

def run(rng, name, n, C=1.0, ngram=(1, 2), max_feat=200000):
    """Train `REPS[n]` models on random subsets of size n; return accuracy, gap, bias, variance."""
    train_acc, test_acc, preds = [], [], []
    for _ in range(REPS[n]):
        idx = rng.choice(len(tr), n, replace=False) if n < len(tr) else np.arange(len(tr))
        vec = TfidfVectorizer(ngram_range=ngram, sublinear_tf=True, min_df=1, max_features=max_feat)
        X = vec.fit_transform(Xtr_text[idx])                  # vocabulary fitted on the subset only
        model = make_model(name, C).fit(X, ytr[idx])
        train_acc.append((model.predict(X) == ytr[idx]).mean())
        p = model.predict(vec.transform(Xte_text)); test_acc.append((p == yte).mean()); preds.append(p)
    P = np.array(preds)
    main = np.apply_along_axis(lambda c: np.bincount(c, minlength=4).argmax(), 0, P)   # modal prediction
    return dict(n=n, train=np.mean(train_acc), test=np.mean(test_acc), test_std=np.std(test_acc),
                gap=np.mean(train_acc) - np.mean(test_acc),
                bias=(main != yte).mean(),            # error of the main prediction
                var=(P != main).mean())               # disagreement with the main prediction

def run_all_experiments():
    rng = np.random.RandomState(42); res = {}
    for name, C in [("LogReg", 10.0), ("LinearSVM", 0.5), ("NaiveBayes", 1)]:
        res[name] = []
        for n in SIZES:
            t = time.time(); r = run(rng, name, n, C); res[name].append(r)
            print(f"{name:10s} n={n:6d}  train={r['train']*100:6.2f}  test={r['test']*100:6.2f}  "
                  f"bias={r['bias']*100:5.2f}  var={r['var']*100:5.2f}  ({time.time()-t:.0f}s)", flush=True)
    res["C_sweep"] = []                                       # regularisation sweep, n = 2000
    for C in [0.01, 0.1, 1, 10, 100, 1000]:
        r = run(rng, "LogReg", 2000, C); r["C"] = C; res["C_sweep"].append(r)
        print(f"C={C:<7} train={r['train']*100:6.2f} test={r['test']*100:6.2f}", flush=True)
    res["F_sweep"] = []                                       # vocabulary-size sweep, n = 2000
    for mf in [100, 500, 2000, 10000, 50000, 200000]:
        r = run(rng, "LogReg", 2000, 10.0, (1, 2), mf); r["maxf"] = mf; res["F_sweep"].append(r)
        print(f"max_features={mf:<7} train={r['train']*100:6.2f} test={r['test']*100:6.2f}", flush=True)
    json.dump(res, open("out/results.json", "w"), indent=1, default=float)
    return res

if "--use-saved" in sys.argv and os.path.exists("out/results.json"):
    res = json.load(open("out/results.json")); print("Loaded saved out/results.json")
else:
    res = run_all_experiments()

# ----------------------------------------------------------------- 3. FIGURES
plt.rcParams.update({"font.size": 10, "axes.grid": True, "grid.alpha": .3,
                     "axes.spines.top": False, "axes.spines.right": False})
N = [x["n"] for x in res["LogReg"]]
g = lambda k, q: np.array([x[q] for x in res[k]]) * 100
XL = "Training-set size (documents, log scale)"

# Figure 4.1  learning curve
f, a = plt.subplots(figsize=(6.2, 3.8))
a.plot(N, g("LogReg", "train"), "o-", c="#c0392b", label="Training accuracy")
a.plot(N, g("LogReg", "test"), "s-", c="#2471a3", label="Test accuracy")
a.fill_between(N, g("LogReg", "test"), g("LogReg", "train"), color="#f5b7b1", alpha=.4, label="Generalisation gap")
a.axhline(90, ls="--", c="grey", lw=1); a.text(110, 90.6, "90% target", fontsize=8, color="grey")
a.set_xscale("log"); a.set_xlabel(XL); a.set_ylabel("Accuracy (%)"); a.set_ylim(50, 102); a.legend(loc="center right", fontsize=8)
f.tight_layout(); f.savefig("out/fig_learning.png", dpi=200); plt.close()

# Figure 4.2  bias and variance
f, a = plt.subplots(figsize=(6.2, 3.8))
a.plot(N, g("LogReg", "bias"), "o-", c="#1f618d", label="Bias (error of main prediction)")
a.plot(N[:-1], g("LogReg", "var")[:-1], "s-", c="#d68910", label="Variance (disagreement with main prediction)")
a.set_xscale("log"); a.set_xlabel(XL); a.set_ylabel("Rate (%)"); a.legend(fontsize=8)
f.tight_layout(); f.savefig("out/fig_biasvar.png", dpi=200); plt.close()

# Figure 4.3  model comparison
f, a = plt.subplots(figsize=(6.2, 3.8))
for k, c, m, l in [("LogReg", "#2471a3", "o", "Logistic Regression"), ("LinearSVM", "#1e8449", "s", "Linear SVM"),
                   ("NaiveBayes", "#8e44ad", "^", "Multinomial Naive Bayes")]:
    a.plot(N, g(k, "test"), m + "-", c=c, label=l)
a.axhline(90, ls="--", c="grey", lw=1)
a.set_xscale("log"); a.set_xlabel(XL); a.set_ylabel("Test accuracy (%)"); a.legend(fontsize=8, loc="lower right")
f.tight_layout(); f.savefig("out/fig_models.png", dpi=200); plt.close()

# Figure 4.4  complexity sweeps
f, ax = plt.subplots(1, 2, figsize=(7.2, 3.5))
C = [x["C"] for x in res["C_sweep"]]
ax[0].plot(C, [x["train"]*100 for x in res["C_sweep"]], "o-", c="#c0392b", label="Train")
ax[0].plot(C, [x["test"]*100 for x in res["C_sweep"]], "s-", c="#2471a3", label="Test")
ax[0].set_xscale("log"); ax[0].set_xlabel("Inverse regularisation C"); ax[0].set_ylabel("Accuracy (%)")
ax[0].set_title("(a) Regularisation strength", fontsize=9); ax[0].legend(fontsize=8)
F = [x["maxf"] for x in res["F_sweep"]]
ax[1].plot(F, [x["train"]*100 for x in res["F_sweep"]], "o-", c="#c0392b", label="Train")
ax[1].plot(F, [x["test"]*100 for x in res["F_sweep"]], "s-", c="#2471a3", label="Test")
ax[1].set_xscale("log"); ax[1].set_xlabel("Vocabulary size (max features)")
ax[1].set_title("(b) Feature-space size", fontsize=9); ax[1].legend(fontsize=8)
f.tight_layout(); f.savefig("out/fig_sweeps.png", dpi=200); plt.close()

# Figures 1.1 and 3.1  flowcharts
def flow(path, steps, w):
    f, a = plt.subplots(figsize=(w, 1.5)); a.axis("off"); a.set_xlim(0, len(steps) * 1.6); a.set_ylim(0, 1)
    for i, s in enumerate(steps):
        x = i * 1.6 + .05
        a.add_patch(FancyBboxPatch((x, .2), 1.25, .6, boxstyle="round,pad=0.02", fc="#d6eaf8", ec="#1f618d"))
        a.text(x + .625, .5, s, ha="center", va="center", fontsize=7.2)
        if i < len(steps) - 1:
            a.annotate("", xy=(x + 1.6, .5), xytext=(x + 1.3, .5), arrowprops=dict(arrowstyle="->", color="#1f618d"))
    f.savefig(path, dpi=200, bbox_inches="tight"); plt.close()
flow("out/fig_flow.png", ["AG News\ncorpus", "Random\nsubsample\nof size n", "TF-IDF\n(1-2 grams)", "Train\nclassifier",
                          "Evaluate on\n7,600 test\ndocs", "Repeat &\nanalyse bias\nvs variance"], 7.4)
flow("out/fig_method.png", ["Requirements\nAnalysis", "Experiment\nDesign", "Implementation\n(Python)",
                            "Experiments\n& Testing", "Analysis &\nReporting"], 6.4)

# Figure 4.5  confusion matrix of the best model (Linear SVM, full training data)
vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, max_features=200000)
X = vec.fit_transform(Xtr_text); best = LinearSVC(C=0.5).fit(X, ytr)
p = best.predict(vec.transform(Xte_text))
names = ["World", "Sports", "Business", "Sci/Tech"]
cm = confusion_matrix(yte, p); rep = classification_report(yte, p, target_names=names, output_dict=True)
json.dump({"cm": cm.tolist(), "rep": rep, "vocab": len(vec.vocabulary_)}, open("out/final.json", "w"))
f, a = plt.subplots(figsize=(4.4, 3.8)); a.grid(False); a.imshow(cm, cmap="Blues")
a.set_xticks(range(4)); a.set_xticklabels(names, fontsize=8); a.set_yticks(range(4)); a.set_yticklabels(names, fontsize=8)
a.set_xlabel("Predicted"); a.set_ylabel("Actual")
for i in range(4):
    for j in range(4):
        a.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=9)
f.tight_layout(); f.savefig("out/fig_cm.png", dpi=200); plt.close()

# ----------------------------------------------------------------- 4. RESULT TABLES
pc = lambda x: f"{x*100:.2f}"
print("\n" + "="*78 + "\nTable 4.1  Learning curve - Logistic Regression\n" + "="*78)
print(f"{'n':>8} {'reps':>5} {'train%':>8} {'test%':>8} {'std':>6} {'gap':>7}")
for x in res["LogReg"]: print(f"{x['n']:>8} {REPS[x['n']]:>5} {pc(x['train']):>8} {pc(x['test']):>8} {pc(x['test_std']):>6} {pc(x['gap']):>7}")
print("\nTable 4.2  Bias-variance decomposition - Logistic Regression")
print(f"{'n':>8} {'bias%':>8} {'var%':>8}  larger term")
for i, x in enumerate(res["LogReg"]):
    v = "n/a (1 run)" if i == 9 else pc(x["var"])
    print(f"{x['n']:>8} {pc(x['bias']):>8} {v:>8}  {'-' if i==9 else ('Variance' if x['var']>x['bias'] else 'Bias')}")
print("\nTable 4.3  Test accuracy (%) of the three classifiers")
print(f"{'n':>8} {'LogReg':>8} {'LinSVM':>8} {'NaiveB':>8}")
for i, n in enumerate(N):
    print(f"{n:>8} {pc(res['LogReg'][i]['test']):>8} {pc(res['LinearSVM'][i]['test']):>8} {pc(res['NaiveBayes'][i]['test']):>8}")
print("\nTable 4.4  Complexity sweeps (n = 2000, Logistic Regression)")
print(f"{'setting':>18} {'train%':>8} {'test%':>8} {'gap':>7} {'bias%':>7} {'var%':>7}")
for x in res["C_sweep"]: print(f"{'C = '+str(x['C']):>18} {pc(x['train']):>8} {pc(x['test']):>8} {pc(x['gap']):>7} {pc(x['bias']):>7} {pc(x['var']):>7}")
for x in res["F_sweep"]: print(f"{'max feat = '+str(x['maxf']):>18} {pc(x['train']):>8} {pc(x['test']):>8} {pc(x['gap']):>7} {pc(x['bias']):>7} {pc(x['var']):>7}")
print("\nTable 4.5  Per-class performance (Linear SVM, full data)")
print(f"{'class':>14} {'prec%':>8} {'recall%':>8} {'F1%':>8}")
for k in names + ["macro avg"]: print(f"{k:>14} {pc(rep[k]['precision']):>8} {pc(rep[k]['recall']):>8} {pc(rep[k]['f1-score']):>8}")
print("Confusion matrix (rows = actual, cols = predicted):\n", cm)
print("\nTable 4.6  90% accuracy target")
for key, lab in [("LinearSVM", "Linear SVM"), ("LogReg", "Logistic Regression"), ("NaiveBayes", "Naive Bayes")]:
    first = next((x["n"] for x in res[key] if x["test"] >= 0.9), "not reached")
    print(f"{lab:>20}: full-data test accuracy = {pc(res[key][9]['test'])}%   first n with >=90%: {first}")
print(f"\nBEST MODEL (Linear SVM) TEST ACCURACY: {pc(rep['accuracy'])}%   macro F1: {pc(rep['macro avg']['f1-score'])}%")
print("Figures saved in the 'out' folder: fig_learning, fig_biasvar, fig_models, fig_sweeps, fig_cm, fig_flow, fig_method (.png)")


Train: 120000  Test: 7600
LogReg     n=   100  train=100.00  test= 58.49  bias=20.92  var=36.42  (17s)
LogReg     n=   200  train=100.00  test= 69.76  bias=18.70  var=23.57  (21s)
LogReg     n=   500  train=100.00  test= 79.68  bias=15.04  var=12.81  (31s)
LogReg     n=  1000  train=100.00  test= 83.20  bias=14.08  var= 8.82  (28s)
LogReg     n=  2000  train=100.00  test= 85.50  bias=13.34  var= 6.10  (41s)
LogReg     n=  5000  train=100.00  test= 87.55  bias=11.62  var= 4.15  (69s)
LogReg     n= 10000  train= 99.98  test= 88.95  bias=10.67  var= 3.02  (59s)
LogReg     n= 20000  train= 99.95  test= 89.97  bias= 9.66  var= 2.59  (95s)
LogReg     n= 50000  train= 99.62  test= 91.28  bias= 8.91  var= 2.06  (83s)
LogReg     n=120000  train= 98.44  test= 92.32  bias= 7.68  var= 0.00  (64s)
LinearSVM  n=   100  train=100.00  test= 58.55  bias=22.37  var=35.76  (10s)
LinearSVM  n=   200  train=100.00  test= 71.18  bias=18.11  var=22.05  (11s)
LinearSVM  n=   500  train=100.00  test= 79.73  bi